# 17 — Orchestration & DAGs

### Why this exists
Orchestration is dependency management plus operational semantics. This lab schedules a tiny DAG, measures parallelism, then introduces a dependency cycle.

### Learning objectives
represent a DAG; topologically schedule tasks; measure critical path; detect cycles

### Prerequisite
Python lists, sets and queue reasoning

### Engineering loop
**Predict → Build → Observe → Break → Diagnose → Improve → Generalize → Defend**

## 🖊️ Sketch note
![Sketch note](../assets/sketch-notes/17-orchestration-dags.svg)

> **Question to carry through the lab:** Why can two tasks run at the same time even though the workflow is ordered? What makes a dependency graph invalid?

## Capability contract

**Capability:** C04 — Lineage Orchestration & Recovery  
**Workstream:** Revenue Data Platform  
**Primary roles:** Data Engineer|Analytics Engineer|Platform Engineer  
**You will prove:** Trace a failure through a workflow and design safe rerun or recovery behavior.


## Mental model
A DAG expresses what must be true before work starts. The scheduler decides when runnable tasks execute; retries, idempotency and backfills are separate semantics that the DAG cannot magically guarantee.

## 1. Predict before you run
Write down what you expect and what evidence would prove you wrong. Do not change the experiment after seeing the result.

## 2. Build
The first cell creates the smallest system that can answer the question. The later cells change one boundary at a time.

## Agent-native lens

**Agent can do:** routine implementation or a first-pass analysis.

**You must understand:** the mechanism and its assumptions.

**You must verify:** use tests, measurements or adversarial data that could prove the output wrong.

**You must decide:** the trade-off, failure response or production boundary.


## Apprenticeship bridge

Use this notebook to prepare for the following Acme Commerce work scenario(s). The scenario gives you symptoms and constraints; it does **not** give you the diagnosis.

- **S04 — Incremental and idempotent pipelines (DATA-1070)** → [`../work_simulations/04_acme_commerce_duplicate_rerun/README.md`](../work_simulations/04_acme_commerce_duplicate_rerun/README.md)
- **S05 — Metadata and lineage (DATA-1081)** → [`../work_simulations/05_acme_commerce_lineage_gap/README.md`](../work_simulations/05_acme_commerce_lineage_gap/README.md)
- **S06 — Retries and backfills (DATA-1094)** → [`../work_simulations/06_acme_commerce_retry_side_effect/README.md`](../work_simulations/06_acme_commerce_retry_side_effect/README.md)

**How to use the bridge:** finish the experiment and independent challenge here, then enter the ticket without reopening the notebook as an answer key. Bring your own prediction, evidence and verification plan.


In [ ]:
tasks={
 "extract":set(), "validate":{"extract"}, "orders":{"validate"},
 "payments":{"validate"}, "publish":{"orders","payments"}
}

def schedule_levels(tasks):
    remaining={k:set(v) for k,v in tasks.items()}; levels=[]
    while remaining:
        ready=sorted(k for k,v in remaining.items() if not v)
        if not ready: raise ValueError("cycle detected")
        levels.append(ready)
        for k in ready: remaining.pop(k)
        for deps in remaining.values(): deps.difference_update(ready)
    return levels

levels=schedule_levels(tasks); print(levels)

In [ ]:
durations={"extract":4,"validate":2,"orders":5,"payments":3,"publish":1}
critical=sum(max(durations[t] for t in level) for level in levels)
serial=sum(durations.values())
print({"serial_minutes":serial,"critical_path_minutes":critical,"parallelism_gain":serial/critical})

In [ ]:
# Break: create a cycle and require the scheduler to reject it.
broken={k:set(v) for k,v in tasks.items()}
broken["extract"]={"publish"}
try: schedule_levels(broken)
except ValueError as exc: print("diagnosed:",exc)

In [ ]:
# Improve: explicit cycle check before registration.
def validate_dag(tasks):
    try: schedule_levels(tasks); return {"valid":True}
    except ValueError: return {"valid":False,"reason":"cycle"}
print(validate_dag(tasks)); print(validate_dag(broken))

## 3. Measure
Do not stop at “it worked”. Record a correctness signal, a failure signal, and at least one quantitative measurement where the concept permits it.

## 4. Break deliberately
Introduce one controlled failure. Keep the failure reproducible so another engineer can reproduce the observation.

## 5. Diagnose
Use evidence, not the implementation you expected. State: **symptom → mechanism → evidence → boundary**. If two explanations fit, design one more measurement.

## 6. Improve
Make the smallest change that addresses the mechanism. Re-run the original experiment and the failure case.

## 7. Generalize
Ask what changes at 10× load, with retries, multiple writers/readers, partial failure, or changing contracts. Separate what this toy proves from what it only illustrates.

## 8. Production bridge
Production orchestration requires idempotency, retries, backfills, concurrency limits, SLAs and observability; dependency order is only the first layer.

## 9. Independent challenge
Add a backfill for only the orders branch. Decide which tasks can safely rerun and which need idempotency keys before the scheduler can execute it.

### Evidence to keep
DAG levels, critical-path calculation, cycle failure, and a written retry/backfill policy

## 10. Explain it in 60–90 seconds
Explain the mechanism, your measurement, the failure you injected, the diagnosis, and the production decision you would make. Avoid tool names until the mental model is clear.